# Create Case Insensitive Fabric Data Warehouse
Unfortunately, there is no means to do an update of an existing Fabric Data Warehouse because the collation setting (which determines case sensitivity) is immutable

In [1]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.keyvault.secrets import SecretClient
import os
from dotenv import load_dotenv
import notebookutils

In [8]:
def create_dw_no_case_sensitivity(dw_display_name:str, workspace_id:str, token:str):
    """
    Function to create a Fabric Data Warehouse that does not have case sensitivity
    Note that the spn running this (the token) will need to have Warehouse.ReadWrite.All API Permission


    dw_display_name:str: name of the data warehouse to be created
    workspace_id:str: GUID of the workspace to create the Fabric Data Warehouse in
    token:str: access token to be used in API Post Header

    """
    url = f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items'

    headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
    }    

    payload = {
  "type": "Warehouse",
  "displayName": dw_display_name,
  "creationPayload": {
    "defaultCollation": "Latin1_General_100_CI_AS_KS_WS_SC_UTF8"
        }
    }

    response = requests.post(url, headers=headers, json=payload)
    print(response.status_code, response.json())


In [9]:
# load env file from lakehouse
# note this way requires a .env file where these can be pulled which has secrets in clear text so shouldn't be used in production

env_path = '/lakehouse/default/Files/.tempenv'
load_dotenv(dotenv_path=env_path)

client_id = os.getenv('AZURE_CLIENT_ID')
tenant_id = os.getenv('AZURE_TENANT_ID')
subscription_id = os.getenv('SUBSCRIPTION_ID')
client_secret = os.getenv('AZURE_CLIENT_SECRET')

kv_uri = "https://kvfabricprodeus2rh.vault.azure.net/"
# credential = DefaultAzureCredential()
# kv_client = SecretClient(vault_url=kv_uri, credential=credential)

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
scope = 'https://analysis.windows.net/powerbi/api/.default'
token = credential.get_token(scope).token

In [21]:
# This is the more secure way that should be used in production

kv_uri = "https://kvfabricprodeus2rh.vault.azure.net/"

client_id_secret = 'fuam-spn-client-id'
tenant_id_secret = 'fuam-spn-tenant-id'
subscription_id_secret = 'subscription-id'
client_secret_name = 'fuam-spn-secret'


client_id = notebookutils.credentials.getSecret(kv_uri, client_id_secret)
tenant_id = notebookutils.credentials.getSecret(kv_uri, tenant_id_secret)
subscription_id = notebookutils.credentials.getSecret(kv_uri, subscription_id_secret)
client_secret = notebookutils.credentials.getSecret(kv_uri, client_secret_name)

new_credential = ClientSecretCredential(tenant_id, client_id, client_secret)
new_scope = 'https://analysis.windows.net/powerbi/api/.default'
new_token = credential.get_token(scope).token

In [20]:
workspace_id = '3cc4f8ca-e289-41aa-abce-5cac216aa369'
dw_name = 'test-case-sens-3'


create_dw_no_case_sensitivity(dw_name, workspace_id, new_token)

202 None
